In [2]:
import sys
!{sys.executable} -m pip install chromadb

import json
import os
import chromadb
from chromadb.utils import embedding_functions

# 1. Setup paths and context-specific names
file_name = "chapter7_activities.json"
context_name = "chapter7_activities"

# Structure: project_root -> data_embedding -> chapter7_activities_db
base_dir = "data_embedding"
db_folder_name = f"{context_name}_db"
db_path = os.path.join(base_dir, db_folder_name)

# Ensure the directory exists
os.makedirs(db_path, exist_ok=True)

# 2. Load the JSON data
with open(file_name, 'r') as f:
    json_data = json.load(f)

# 3. Initialize Chroma Client for this specific directory
# This generates chroma.sqlite3 and the vector folders inside db_path
client = chromadb.PersistentClient(path=db_path)

# 4. Define the Embedding Function
default_ef = embedding_functions.DefaultEmbeddingFunction()

# 5. Create a collection specific to this chapter
chapter7_activities_collection = client.get_or_create_collection(
    name=f"{context_name}_embeddings",
    embedding_function=default_ef
)

# 6. Prepare data for storage
# We use 'text' as the document and map 'type', 'activity_number', and 'page' to metadata
documents = [item["text"] for item in json_data]
metadatas = [
    {
        "type": item["type"],
        "activity_number": item["activity_number"],
        "page": item["page"],
        "source_file": file_name
    } for item in json_data
]
ids = [f"{context_name}_id_{i}" for i in range(len(json_data))]

# 7. Generate and store embeddings
chapter7_activities_collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

# 8. Create the 'source_snapshot.json' in the same folder as requested
snapshot_path = os.path.join(db_path, "source_snapshot.json")
with open(snapshot_path, 'w') as f:
    json.dump(json_data, f, indent=4)

# 9. Retrieve and store result in the context-specific variable
chapter7_activities_output = chapter7_activities_collection.get(
    include=['embeddings', 'documents', 'metadatas']
)

# 10. Verification and Inspection
raw_embeddings = chapter7_activities_output.get('embeddings')

if raw_embeddings is not None and len(raw_embeddings) > 0:
    print(f"--- Process Complete for {context_name} ---")
    print(f"Folder Created: {db_path}")
    print(f"Total Activities Embedded: {len(raw_embeddings)}")

    # Print sample of the first activity and its vector
    print("\nSample Check:")
    print(f"ID: {chapter7_activities_output['ids'][0]}")
    print(f"Text Preview: {chapter7_activities_output['documents'][0][:50]}...")
    print(f"Vector (First 5 dims): {raw_embeddings[0][:5]}")
else:
    print("Error: Embeddings could not be retrieved.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.2 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 53.5MiB/s]


--- Process Complete for chapter7_activities ---
Folder Created: data_embedding/chapter7_activities_db
Total Activities Embedded: 11

Sample Check:
ID: chapter7_activities_id_0
Text Preview: Activity 7.1
Discuss whether the walls of your
cla...
Vector (First 5 dims): [ 0.05904596 -0.01189282  0.01926256  0.00051335  0.00331445]


In [4]:
import shutil

# Define the output zip file name
zip_file_name = os.path.join(base_dir, f"{context_name}_db.zip")

# Create a zip archive of the db_path directory
shutil.make_archive(db_path, 'zip', db_path)

print(f"Vector database compressed to: {zip_file_name}. You can download this file from the Colab file browser.")

Vector database compressed to: data_embedding/chapter7_activities_db.zip. You can download this file from the Colab file browser.
